# 두번째 모델 생성  

첫번째 모델 생성 후 로보 플로우 사이트의 wtbd 데이터셋으로 테스트해보니  

재현율이 현저히 떨어지고 모델이 오염 또는 손상 탐지 하는 성능이 기대 이하로 떨어지는 부분 발견함  

팀원들과 논의 끝에 학습 시점의 데이터를 증강 시켜 좀 더 많은 데이터를 학습하게 하자는  

결론을 내고 다시 모델 학습으로 돌아가 데이터 증강 후 학습함  


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ajifoster3/yolo-annotated-wind-turbines-586x371")

print("Path to dataset files:", path)

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("Supported arch list:", torch.cuda.get_arch_list())

In [ ]:
"""
YOLOv8s recall-focused training for MQ02 Wind Turbine Damage / Pollution Detection.

Patch goal
----------
This version changes the previous fully 5:5 augmented dataset strategy into the
learning method recommended for this dataset:

1) Use a fresh basic YOLOv8s pretrained model: YOLO("yolov8s.pt").
   - This is transfer learning from the official basic YOLOv8s weights.
   - It does NOT continue from your previously trained best.pt.
   - If you truly want random-weight scratch training, change MODEL_ARCH to "yolov8s.yaml".

2) Split original images first to prevent leakage.
   - Original abnormal labeled images are split into train / val / test.
   - Original normal empty-label/unlabeled images are split into train / val / test.
   - Augmented train images are generated only from train abnormal sources.

3) Train set design:
   - Keep original abnormal train images.
   - Add augmented abnormal train images until TRAIN_ABNORMAL_TOTAL_TARGET is reached.
   - Limit normal/background train images to TRAIN_NORMAL_RATIO_TO_ABNORMAL.
   - Default: abnormal target 12,000 and normal target 20% = about 2,400 normals.
   - This avoids training the model to become too conservative and miss defects.

4) Validation/test design:
   - No augmentation in val/test.
   - Val/test use original abnormal images + original normal background images only.
   - Default val/test are balanced 1:1 abnormal:normal.
   - A separate test_real_distribution folder is also created using all test normals.

5) Abnormal augmentation balance:
   - Augmented abnormal train images are sampled 50% from dirt/contamination sources
     and 50% from damage sources when possible.

6) Output:
   - data.yaml for training/evaluation.
   - dataset_summary.csv.
   - train loss graph.
   - validation loss graph.
   - precision/recall graph.
   - mAP50/mAP50-95 graph.
   - final balanced test evaluation.
   - final real-distribution test evaluation.
   - optional normal-only false-positive check.
"""

import sys
import random
import shutil
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np


# ============================================================
# 1. Settings
# ============================================================

SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# Works in Kaggle, Colab, and local environments.
if Path("/kaggle/working").exists():
    WORK_DIR = Path("/kaggle/working")
elif Path("/content").exists():
    WORK_DIR = Path("/content")
else:
    WORK_DIR = SCRIPT_DIR

# You confirmed this path can appear after kagglehub download in Colab/Kaggle-like sessions.
FIXED_KAGGLE_DATASET_PATH = Path("/kaggle/input/yolo-annotated-wind-turbines-586x371")
DATASET_SLUG = "ajifoster3/yolo-annotated-wind-turbines-586x371"

if FIXED_KAGGLE_DATASET_PATH.exists():
    SRC = FIXED_KAGGLE_DATASET_PATH.resolve()
    print("Using fixed dataset path:", SRC)
else:
    try:
        import kagglehub
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub

    SRC = Path(kagglehub.dataset_download(DATASET_SLUG)).resolve()
    print("Fixed path not found. kagglehub dataset path:", SRC)

print("Dataset SRC:", SRC)
print("Working/output directory:", WORK_DIR)

# Model start point.
# Recommended for this dataset size: pretrained basic YOLOv8s.
# This downloads/uses the official YOLOv8s file, not your previously trained checkpoint.
MODEL_ARCH = "yolov8s.pt"

# New output names. These avoid mixing with the old 5:5 full-balanced dataset/run.
OUT = str(WORK_DIR / "wind_turbine_recall_focused_yolov8s_dataset")
PROJECT_DIR = str(WORK_DIR / "runs")
RUN_NAME = "yolov8s_recall_focused_train_aug_only"

# Resume control.
# If training stops because Colab disconnects or the cell is interrupted, rerun this script.
# When /content/runs/<RUN_NAME>/weights/last.pt exists, training resumes from that checkpoint
# instead of starting again from epoch 1.
AUTO_RESUME_IF_LAST_EXISTS = True
FORCE_RESTART_TRAINING = False
# Optional: set a custom checkpoint path string here if needed, for example:
# RESUME_CHECKPOINT_PATH = "/content/runs/yolov8s_recall_focused_train_aug_only/weights/last.pt"
RESUME_CHECKPOINT_PATH = None

# Training control.
EPOCHS = 35   # 640 is better than 512 for small turbine damage/pollution regions.
BATCH = 16        # Fixed batch. Safer than AutoBatch on Colab T4 for this setup.
SEED = 42
WORKERS = 2

# Source split ratio for original pools.
RATIO = (0.6, 0.2, 0.2)
SPLIT_NAMES = ("train", "val", "test")

# Dataset design recommended for recall-sensitive defect detection.
# This target includes original abnormal train images + augmented abnormal train images.
TRAIN_ABNORMAL_TOTAL_TARGET = 12000

# Use limited normal backgrounds in train. Too many normals can reduce recall.
TRAIN_NORMAL_RATIO_TO_ABNORMAL = 0.20
MIN_TRAIN_NORMAL_IMAGES = 500
MAX_TRAIN_NORMAL_IMAGES = 3000

# Val/test are original-only and balanced by default.
EVAL_NORMAL_TO_ABNORMAL_RATIO = 1.00
CREATE_REAL_DISTRIBUTION_TEST = True
CREATE_NORMAL_ONLY_TEST = True

# Optional normal-only false positive check after training.
# The previous version evaluated many normal images at once and could trigger CUDA OOM
# after training/validation had already filled GPU memory. This version uses tiny batches,
# clears CUDA cache regularly, and never lets this optional check crash the notebook.
RUN_NORMAL_FP_EVAL = True
NORMAL_FP_EVAL_MAX_IMAGES = 1000
NORMAL_FP_CONF = 0.25
NORMAL_FP_BATCH_SIZE = 1
NORMAL_FP_CLEAR_CACHE_EVERY = 25
SKIP_NORMAL_FP_ON_OOM = True

# Safety export.
# Artifacts are copied and zipped immediately after training and again after each evaluation stage.
# This protects best.pt/last.pt/graphs/results.csv even if a later optional step fails.
EXPORT_ARTIFACTS = True
EXPORT_DIR = str(WORK_DIR / "yolov8s_recall_focused_export")
EXPORT_ZIP_BASENAME = str(WORK_DIR / "yolov8s_recall_focused_export")

# Dataset convention.
# 0 = dirt / contamination / pollution
# 1 = damage
DIRT_CLASS_ID = 0
DAMAGE_CLASS_ID = 1
CLASS_NAMES = {
    DIRT_CLASS_ID: "dirt",
    DAMAGE_CLASS_ID: "damage",
}

# Abnormal train augmentation source balance.
ABNORMAL_CLASS_RATIO = {
    DIRT_CLASS_ID: 0.50,
    DAMAGE_CLASS_ID: 0.50,
}

# Set True when you intentionally want to delete and regenerate all generated data.
REBUILD_DATASET = False

# Checkpoint / early stopping.
EARLY_STOP_PATIENCE = 25
SAVE_PERIOD = 1
FREEZE_LAYERS = 0

# Albumentations bbox safety.
AUGMENT_OUTPUT_EXT = ".jpg"
AUGMENT_JPEG_QUALITY = 95
AUGMENT_MAX_RETRY = 50
AUGMENT_MIN_VISIBILITY = 0.10
AUGMENT_MIN_AREA = 4

IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".bmp", ".webp"]


# ============================================================
# 2. Package check
# ============================================================

def pip_install(package_args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + package_args
    print("Installing:", " ".join(package_args))
    subprocess.check_call(cmd)


def check_packages():
    try:
        import torch
        print("torch OK:", torch.__version__)
    except Exception as exc:
        raise RuntimeError("torch import failed. Install torch first.") from exc

    try:
        import ultralytics
        print("ultralytics OK:", ultralytics.__version__)
    except Exception:
        pip_install(["ultralytics"])
        import ultralytics
        print("ultralytics installed:", ultralytics.__version__)

    try:
        import albumentations
        print("albumentations OK:", albumentations.__version__)
    except Exception:
        pip_install(["albumentations"])
        import albumentations
        print("albumentations installed:", albumentations.__version__)

    try:
        import pandas
        print("pandas OK:", pandas.__version__)
    except Exception:
        pip_install(["pandas"])
        import pandas
        print("pandas installed:", pandas.__version__)

    try:
        import matplotlib
        print("matplotlib OK:", matplotlib.__version__)
    except Exception:
        pip_install(["matplotlib"])
        import matplotlib
        print("matplotlib installed:", matplotlib.__version__)

    try:
        import cv2 as _cv2
        print("opencv OK:", _cv2.__version__)
    except Exception:
        pip_install(["opencv-python-headless"])
        import cv2 as _cv2
        print("opencv installed:", _cv2.__version__)


def get_device():
    import torch
    if torch.cuda.is_available():
        print("CUDA available. Using GPU 0.")
        return 0
    print("CUDA not available. Using CPU. This will be slower.")
    return "cpu"


# ============================================================
# 3. Dataset search
# ============================================================

def get_image_files(img_dir):
    img_dir = Path(img_dir)
    images = []
    for ext in IMAGE_EXTS:
        images.extend(img_dir.glob(f"*{ext}"))
        images.extend(img_dir.glob(f"*{ext.upper()}"))
    return sorted(set(images), key=lambda p: p.stem)


def count_labeled_pairs(root):
    root = Path(root)
    img_dir = root / "images"
    lbl_dir = root / "labels"
    if not img_dir.exists() or not lbl_dir.exists():
        return 0
    image_stems = set(p.stem for p in get_image_files(img_dir))
    label_stems = set(p.stem for p in lbl_dir.glob("*.txt") if p.name.lower() != "labels.txt")
    return len(image_stems & label_stems)


def count_total_images(root):
    img_dir = Path(root) / "images"
    return len(get_image_files(img_dir)) if img_dir.exists() else 0


def candidate_roots_from_base(base_path):
    base = Path(base_path).expanduser().resolve()
    candidates = [
        base,
        base / "NordTank586x371",
        base / "NordTank586x371" / "NordTank586x371",
    ]
    if base.exists():
        for p in base.rglob("*"):
            if p.is_dir() and (p / "images").exists() and (p / "labels").exists():
                candidates.append(p)

    unique = []
    seen = set()
    for c in candidates:
        c = Path(c).expanduser().resolve()
        if str(c) not in seen:
            seen.add(str(c))
            unique.append(c)
    return unique


def find_dataset_root():
    best_path = None
    best_labeled = 0
    best_total = 0
    checked = []

    for candidate in candidate_roots_from_base(SRC):
        labeled = count_labeled_pairs(candidate)
        total = count_total_images(candidate)
        checked.append((candidate, labeled, total))
        if labeled > best_labeled or (labeled == best_labeled and total > best_total):
            best_path, best_labeled, best_total = candidate, labeled, total

    print("Manual SRC:", SRC)
    print("Best nested root:", best_path)
    print("Labeled pairs:", best_labeled)
    print("Total images:", best_total)

    if best_path is not None and best_labeled > 0 and best_total > 0:
        return best_path

    print("Checked paths:")
    for p, labeled, total in checked:
        print(" ", p, "labeled:", labeled, "total_images:", total)

    raise FileNotFoundError("Could not find images/ and labels/ folders inside the dataset.")


# ============================================================
# 4. Label / IO helpers
# ============================================================

def safe_imread(path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def safe_imwrite(path, image, quality=AUGMENT_JPEG_QUALITY):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix.lower() if path.suffix else ".jpg"
    params = []
    if ext in [".jpg", ".jpeg"]:
        params = [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)]
    ok, encoded = cv2.imencode(ext, image, params)
    if not ok:
        raise RuntimeError(f"Failed to encode image: {path}")
    encoded.tofile(str(path))


def safe_copy(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def read_yolo_label(label_path):
    bboxes = []
    class_labels = []
    label_path = Path(label_path)
    if not label_path.exists():
        return bboxes, class_labels

    text = label_path.read_text(encoding="utf-8", errors="ignore").strip()
    if not text:
        return bboxes, class_labels

    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
        except Exception:
            continue

        if bw <= 0 or bh <= 0:
            continue
        if not (0 <= xc <= 1 and 0 <= yc <= 1):
            continue

        bboxes.append([xc, yc, bw, bh])
        class_labels.append(cls_id)

    return bboxes, class_labels


def write_yolo_label(label_path, bboxes, class_labels):
    lines = []
    for bbox, cls_id in zip(bboxes, class_labels):
        xc, yc, bw, bh = bbox
        xc = min(max(float(xc), 0.0), 1.0)
        yc = min(max(float(yc), 0.0), 1.0)
        bw = min(max(float(bw), 0.0), 1.0)
        bh = min(max(float(bh), 0.0), 1.0)
        if bw <= 0 or bh <= 0:
            continue
        lines.append(f"{int(cls_id)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    Path(label_path).parent.mkdir(parents=True, exist_ok=True)
    Path(label_path).write_text("\n".join(lines), encoding="utf-8")


def label_has_class(label_path, class_id):
    _, labels = read_yolo_label(label_path)
    return int(class_id) in [int(x) for x in labels]


def label_instance_count(label_path):
    _, labels = read_yolo_label(label_path)
    return Counter(int(x) for x in labels)


def make_item(base, image_path, label_path, kind):
    if label_path is None:
        return {
            "base": base,
            "image_path": Path(image_path),
            "label_path": None,
            "kind": kind,
            "has_dirt": False,
            "has_damage": False,
            "instance_counts": Counter(),
        }

    bboxes, class_labels = read_yolo_label(label_path)
    counts = Counter(int(x) for x in class_labels)
    return {
        "base": base,
        "image_path": Path(image_path),
        "label_path": Path(label_path),
        "kind": kind,
        "has_dirt": DIRT_CLASS_ID in counts,
        "has_damage": DAMAGE_CLASS_ID in counts,
        "instance_counts": counts,
    }


# ============================================================
# 5. Splitting and sampling
# ============================================================

def split_group(group):
    """Deterministic split for one item group."""
    group = sorted(group, key=lambda x: x["base"])
    random.Random(SEED).shuffle(group)

    n = len(group)
    train_end = int(n * RATIO[0])
    val_end = train_end + int(n * RATIO[1])
    return group[:train_end], group[train_end:val_end], group[val_end:]


def split_labeled_preserve_classes(labeled_items):
    """Split labeled data after reporting class source counts."""
    dirt_items = [x for x in labeled_items if x["has_dirt"]]
    damage_items = [x for x in labeled_items if x["has_damage"]]
    both_items = [x for x in labeled_items if x["has_dirt"] and x["has_damage"]]
    other_items = [x for x in labeled_items if not x["has_dirt"] and not x["has_damage"]]

    labeled_split = split_group(labeled_items)

    out = {}
    for idx, split_name in enumerate(SPLIT_NAMES):
        items = list(labeled_split[idx])
        rng = random.Random(SEED + idx)
        rng.shuffle(items)
        out[split_name] = items

    print("")
    print("Original labeled class source counts")
    print("====================================")
    print("labeled total :", len(labeled_items))
    print("dirt sources  :", len(dirt_items))
    print("damage sources:", len(damage_items))
    print("both sources  :", len(both_items))
    print("other sources :", len(other_items))

    return out


def split_normal(normal_items):
    normal_split = split_group(normal_items)
    out = {}
    for idx, split_name in enumerate(SPLIT_NAMES):
        items = list(normal_split[idx])
        rng = random.Random(SEED + 100 + idx)
        rng.shuffle(items)
        out[split_name] = items
    return out


def sample_items(items, needed_count, rng):
    """Return exactly needed_count items, using replacement only when required."""
    if needed_count <= 0:
        return []
    if not items:
        raise RuntimeError("Cannot sample from an empty item list.")

    shuffled = list(items)
    rng.shuffle(shuffled)

    if needed_count <= len(shuffled):
        return shuffled[:needed_count]

    out = []
    out.extend(shuffled)
    while len(out) < needed_count:
        out.append(rng.choice(shuffled))
    return out[:needed_count]


def compute_train_normal_target(train_abnormal_total, available_train_normals):
    target = int(round(train_abnormal_total * TRAIN_NORMAL_RATIO_TO_ABNORMAL))
    target = max(target, MIN_TRAIN_NORMAL_IMAGES)
    target = min(target, MAX_TRAIN_NORMAL_IMAGES)
    target = min(target, available_train_normals) if available_train_normals > 0 else 0
    return target


def compute_eval_normal_target(abnormal_count, available_normal_count):
    target = int(round(abnormal_count * EVAL_NORMAL_TO_ABNORMAL_RATIO))
    target = min(target, available_normal_count)
    return max(target, 0)


# ============================================================
# 6. Augmentation
# ============================================================

def build_augmentation_pipeline():
    import albumentations as A

    def image_compression():
        try:
            return A.ImageCompression(quality_range=(60, 95), p=1.0)
        except TypeError:
            return A.ImageCompression(quality_lower=60, quality_upper=95, p=1.0)

    def coarse_dropout():
        try:
            return A.CoarseDropout(
                num_holes_range=(1, 6),
                hole_height_range=(0.02, 0.08),
                hole_width_range=(0.02, 0.08),
                fill=0,
                p=1.0,
            )
        except TypeError:
            return A.CoarseDropout(
                max_holes=6,
                max_height=32,
                max_width=32,
                min_holes=1,
                fill_value=0,
                p=1.0,
            )

    # Varied but bbox-safe augmentations for camera, weather, illumination,
    # compression, motion, and view-point changes.
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.OneOf(
                [
                    A.Affine(
                        scale=(0.85, 1.18),
                        translate_percent=(-0.07, 0.07),
                        rotate=(-12, 12),
                        shear=(-5, 5),
                        p=1.0,
                    ),
                    A.Perspective(scale=(0.015, 0.050), p=1.0),
                ],
                p=0.75,
            ),
            A.OneOf(
                [
                    A.RandomBrightnessContrast(brightness_limit=0.30, contrast_limit=0.30, p=1.0),
                    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=28, val_shift_limit=28, p=1.0),
                    A.RandomGamma(gamma_limit=(75, 135), p=1.0),
                    A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=1.0),
                    A.Sharpen(alpha=(0.12, 0.40), lightness=(0.80, 1.20), p=1.0),
                ],
                p=0.90,
            ),
            A.OneOf(
                [
                    A.MotionBlur(blur_limit=5, p=1.0),
                    A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                    A.GaussNoise(p=1.0),
                    image_compression(),
                ],
                p=0.55,
            ),
            A.OneOf(
                [
                    A.RandomShadow(p=1.0),
                    coarse_dropout(),
                ],
                p=0.20,
            ),
        ],
        bbox_params=A.BboxParams(
            format="yolo",
            label_fields=["class_labels"],
            min_visibility=AUGMENT_MIN_VISIBILITY,
            min_area=AUGMENT_MIN_AREA,
        ),
    )


def build_class_source_pools(labeled_items):
    pools = {
        DIRT_CLASS_ID: [x for x in labeled_items if x["has_dirt"]],
        DAMAGE_CLASS_ID: [x for x in labeled_items if x["has_damage"]],
    }
    for cls_id, items in pools.items():
        print(f"  class {cls_id} source images:", len(items))

    if not pools[DIRT_CLASS_ID]:
        print("Warning: no dirt/contamination sources found. Dirt quota will fallback to all labeled train images.")
    if not pools[DAMAGE_CLASS_ID]:
        print("Warning: no damage sources found. Damage quota will fallback to all labeled train images.")

    return pools


def build_abnormal_class_plan(needed_count, rng):
    dirt_count = int(round(needed_count * ABNORMAL_CLASS_RATIO[DIRT_CLASS_ID]))
    dirt_count = min(max(dirt_count, 0), needed_count)
    damage_count = needed_count - dirt_count
    plan = [DIRT_CLASS_ID] * dirt_count + [DAMAGE_CLASS_ID] * damage_count
    rng.shuffle(plan)
    return plan


def create_one_augmented_image(transform, src_item, desired_class_id):
    img = safe_imread(src_item["image_path"])
    if img is None:
        return None

    bboxes, class_labels = read_yolo_label(src_item["label_path"])
    if not bboxes:
        return None

    for _ in range(AUGMENT_MAX_RETRY):
        try:
            out = transform(image=img, bboxes=bboxes, class_labels=class_labels)
        except Exception:
            continue

        aug_boxes = list(out.get("bboxes", []))
        aug_labels = [int(x) for x in out.get("class_labels", [])]
        aug_img = out.get("image", None)

        if aug_img is None or not aug_boxes:
            continue

        # Keep the requested class visible after bbox-safe augmentation.
        if int(desired_class_id) not in aug_labels:
            continue

        return aug_img, aug_boxes, aug_labels

    return None


def create_augmented_train(labeled_items, image_out_dir, label_out_dir, needed_count):
    """Create augmented abnormal images only for the train split."""
    if needed_count <= 0:
        print("No train augmentation needed.")
        return 0
    if not labeled_items:
        raise RuntimeError("No train labeled source images available for augmentation.")

    transform = build_augmentation_pipeline()
    rng = random.Random(SEED + 2026)
    class_pools = build_class_source_pools(labeled_items)
    class_plan = build_abnormal_class_plan(needed_count, rng)

    print("")
    print("Train-only augmented abnormal generation")
    print("========================================")
    print("Original train labeled source images:", len(labeled_items))
    print("Augmented abnormal train images to create:", needed_count)
    print("Requested abnormal source ratio: dirt 50% / damage 50%")

    created = 0
    attempts = 0
    requested_source_counts = {DIRT_CLASS_ID: 0, DAMAGE_CLASS_ID: 0}
    visible_label_counts = {DIRT_CLASS_ID: 0, DAMAGE_CLASS_ID: 0}

    for desired_class_id in class_plan:
        source_pool = class_pools.get(desired_class_id, []) or labeled_items
        success_result = None
        selected_src = None

        for _ in range(AUGMENT_MAX_RETRY):
            attempts += 1
            src_item = rng.choice(source_pool)
            result = create_one_augmented_image(transform, src_item, desired_class_id)
            if result is not None:
                success_result = result
                selected_src = src_item
                break

        if success_result is None:
            # Last fallback: any labeled image with at least one surviving bbox.
            for _ in range(AUGMENT_MAX_RETRY):
                attempts += 1
                src_item = rng.choice(labeled_items)
                img = safe_imread(src_item["image_path"])
                if img is None:
                    continue
                bboxes, class_labels = read_yolo_label(src_item["label_path"])
                if not bboxes:
                    continue
                try:
                    out = transform(image=img, bboxes=bboxes, class_labels=class_labels)
                except Exception:
                    continue
                if out.get("bboxes"):
                    success_result = (
                        out["image"],
                        list(out["bboxes"]),
                        [int(x) for x in out["class_labels"]],
                    )
                    selected_src = src_item
                    break

        if success_result is None or selected_src is None:
            print(f"Warning: failed to create one augmented image for requested class={desired_class_id}")
            continue

        aug_img, aug_boxes, aug_labels = success_result
        created += 1
        requested_source_counts[int(desired_class_id)] += 1

        for cls_id in [DIRT_CLASS_ID, DAMAGE_CLASS_ID]:
            if int(cls_id) in [int(x) for x in aug_labels]:
                visible_label_counts[int(cls_id)] += 1

        base = f"{selected_src['base']}__train__aug_cls{desired_class_id}_{created:06d}"
        aug_img_path = Path(image_out_dir) / f"{base}{AUGMENT_OUTPUT_EXT}"
        aug_lbl_path = Path(label_out_dir) / f"{base}.txt"

        safe_imwrite(aug_img_path, aug_img)
        write_yolo_label(aug_lbl_path, aug_boxes, aug_labels)

        if created % 500 == 0 or created == needed_count:
            print(f"Created augmented train images: {created}/{needed_count}")

    if created != needed_count:
        raise RuntimeError(f"Augmented train count mismatch: created={created}, needed={needed_count}")

    print("Train augmentation finished")
    print("Created:", created)
    print("Attempts:", attempts)
    print("Requested source class counts:", requested_source_counts)
    print("Visible label file class counts:", visible_label_counts)
    return created


# ============================================================
# 7. Dataset preparation helpers
# ============================================================

def collect_items(src_root):
    img_dir = src_root / "images"
    lbl_dir = src_root / "labels"

    all_images = get_image_files(img_dir)
    label_files = sorted(p for p in lbl_dir.glob("*.txt") if p.name.lower() != "labels.txt")
    label_by_stem = {p.stem: p for p in label_files}

    labeled_items = []
    normal_items = []

    for image_path in all_images:
        base = image_path.stem
        label_path = label_by_stem.get(base)

        if label_path is None:
            normal_items.append(make_item(base, image_path, None, "normal_empty_label"))
        else:
            bboxes, class_labels = read_yolo_label(label_path)
            if not bboxes:
                normal_items.append(make_item(base, image_path, label_path, "normal_empty_label"))
            else:
                labeled_items.append(make_item(base, image_path, label_path, "abnormal_labeled"))

    print("Total image files:", len(all_images))
    print("Label txt files:", len(label_files))
    print("Abnormal labeled images:", len(labeled_items))
    print("Known-normal empty/unlabeled images:", len(normal_items))

    if not labeled_items:
        raise RuntimeError("No labeled abnormal images found.")
    if not normal_items:
        raise RuntimeError("No normal empty/unlabeled images found.")

    return labeled_items, normal_items


def copy_labeled_items(items, split_name, image_out_dir, label_out_dir, tag="orig"):
    image_out_dir = Path(image_out_dir)
    label_out_dir = Path(label_out_dir)
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    for i, item in enumerate(items, start=1):
        src_img = Path(item["image_path"])
        src_lbl = Path(item["label_path"])
        base = f"{item['base']}__{split_name}__{tag}_{i:06d}"
        dst_img = image_out_dir / f"{base}{src_img.suffix.lower()}"
        dst_lbl = label_out_dir / f"{base}.txt"
        safe_copy(src_img, dst_img)
        safe_copy(src_lbl, dst_lbl)

    return len(items)


def copy_normal_items(items, split_name, image_out_dir, label_out_dir, needed_count, tag="normal"):
    image_out_dir = Path(image_out_dir)
    label_out_dir = Path(label_out_dir)
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    rng = random.Random(SEED + 4040 + {"train": 0, "val": 1, "test": 2, "test_real": 3, "test_normal": 4}.get(split_name, 9))
    selected = sample_items(items, needed_count, rng)

    if needed_count > len(items):
        print(
            f"Warning: split={split_name} needs {needed_count} normal images, "
            f"but only {len(items)} source normals are available. Sampling with replacement."
        )

    for i, item in enumerate(selected, start=1):
        src_img = Path(item["image_path"])
        base = f"{item['base']}__{split_name}__{tag}_{i:06d}"
        dst_img = image_out_dir / f"{base}{src_img.suffix.lower()}"
        dst_lbl = label_out_dir / f"{base}.txt"
        safe_copy(src_img, dst_img)
        dst_lbl.write_text("", encoding="utf-8")

    return len(selected)


def count_empty_label_files(label_dir):
    count = 0
    for p in Path(label_dir).glob("*.txt"):
        if not p.read_text(encoding="utf-8", errors="ignore").strip():
            count += 1
    return count


def count_dataset_label_stats(label_dir):
    stats = {
        "label_files": 0,
        "empty_backgrounds": 0,
        "object_label_files": 0,
        "dirt_files": 0,
        "damage_files": 0,
        "dirt_instances": 0,
        "damage_instances": 0,
    }
    for p in Path(label_dir).glob("*.txt"):
        stats["label_files"] += 1
        bboxes, labels = read_yolo_label(p)
        if not bboxes:
            stats["empty_backgrounds"] += 1
            continue
        stats["object_label_files"] += 1
        labels = [int(x) for x in labels]
        if DIRT_CLASS_ID in labels:
            stats["dirt_files"] += 1
        if DAMAGE_CLASS_ID in labels:
            stats["damage_files"] += 1
        c = Counter(labels)
        stats["dirt_instances"] += c[DIRT_CLASS_ID]
        stats["damage_instances"] += c[DAMAGE_CLASS_ID]
    return stats


def summarize_split(out_path, split_name):
    img_dir = Path(out_path) / split_name / "images"
    lbl_dir = Path(out_path) / split_name / "labels"
    stats = count_dataset_label_stats(lbl_dir)
    stats.update({
        "split": split_name,
        "images": len(list(img_dir.glob("*"))) if img_dir.exists() else 0,
        "image_dir": str(img_dir),
        "label_dir": str(lbl_dir),
    })
    return stats


def write_dataset_summary(out_path):
    import pandas as pd

    split_list = ["train", "val", "test"]
    if CREATE_REAL_DISTRIBUTION_TEST:
        split_list.append("test_real")
    if CREATE_NORMAL_ONLY_TEST:
        split_list.append("test_normal")

    rows = [summarize_split(out_path, s) for s in split_list]
    df = pd.DataFrame(rows)
    summary_path = Path(out_path) / "dataset_summary.csv"
    df.to_csv(summary_path, index=False)
    print("")
    print("Dataset summary")
    print("===============")
    print(df[[
        "split", "images", "label_files", "object_label_files", "empty_backgrounds",
        "dirt_files", "damage_files", "dirt_instances", "damage_instances"
    ]].to_string(index=False))
    print("Saved dataset summary:", summary_path)
    return summary_path


def prepare_recall_focused_dataset():
    out_path = Path(OUT).resolve()
    data_yaml = out_path / "data.yaml"
    real_yaml = out_path / "data_real_test.yaml"

    if out_path.exists() and not REBUILD_DATASET and data_yaml.exists():
        print("Existing recall-focused dataset found. Reusing:", out_path)
        test_count = len(list((out_path / "test" / "images").glob("*")))
        return str(data_yaml), str(real_yaml), test_count

    if out_path.exists():
        shutil.rmtree(out_path)

    src_root = find_dataset_root()
    labeled_items, normal_items = collect_items(src_root)

    labeled_by_split = split_labeled_preserve_classes(labeled_items)
    normal_by_split = split_normal(normal_items)

    train_labeled = labeled_by_split["train"]
    val_labeled = labeled_by_split["val"]
    test_labeled = labeled_by_split["test"]

    train_abnormal_total_target = max(TRAIN_ABNORMAL_TOTAL_TARGET, len(train_labeled))
    train_aug_needed = train_abnormal_total_target - len(train_labeled)
    train_normal_target = compute_train_normal_target(train_abnormal_total_target, len(normal_by_split["train"]))

    val_normal_target = compute_eval_normal_target(len(val_labeled), len(normal_by_split["val"]))
    test_normal_target = compute_eval_normal_target(len(test_labeled), len(normal_by_split["test"]))

    print("")
    print("Recall-focused dataset plan")
    print("============================")
    print("Source split ratio:", RATIO)
    print("Model start:", MODEL_ARCH)
    print("Train abnormal target total:", train_abnormal_total_target)
    print("Train original abnormal:", len(train_labeled))
    print("Train augmented abnormal needed:", train_aug_needed)
    print("Train normal target:", train_normal_target)
    print("Train normal ratio to abnormal:", TRAIN_NORMAL_RATIO_TO_ABNORMAL)
    print("Val original abnormal:", len(val_labeled), "Val normal target:", val_normal_target)
    print("Test original abnormal:", len(test_labeled), "Test normal target:", test_normal_target)
    print("Val/test augmentation: none")
    print("Train abnormal augmentation ratio: dirt 50% / damage 50%")
    print("Real-distribution test enabled:", CREATE_REAL_DISTRIBUTION_TEST)

    # Create train split: original abnormal + augmented abnormal + limited normal.
    train_img_dir = out_path / "train" / "images"
    train_lbl_dir = out_path / "train" / "labels"
    copy_labeled_items(train_labeled, "train", train_img_dir, train_lbl_dir, tag="orig_abnormal")
    create_augmented_train(train_labeled, train_img_dir, train_lbl_dir, train_aug_needed)
    copy_normal_items(normal_by_split["train"], "train", train_img_dir, train_lbl_dir, train_normal_target, tag="limited_normal")

    # Create val/test: original-only, balanced normal background.
    for split_name, labeled_count, normal_target in [
        ("val", len(val_labeled), val_normal_target),
        ("test", len(test_labeled), test_normal_target),
    ]:
        img_dir = out_path / split_name / "images"
        lbl_dir = out_path / split_name / "labels"
        copy_labeled_items(labeled_by_split[split_name], split_name, img_dir, lbl_dir, tag="orig_abnormal")
        copy_normal_items(normal_by_split[split_name], split_name, img_dir, lbl_dir, normal_target, tag="balanced_normal")
        print(f"Created {split_name}: abnormal={labeled_count}, normal={normal_target}, augmentation=0")

    # Real-distribution test: original abnormal test + all available original normal test.
    if CREATE_REAL_DISTRIBUTION_TEST:
        real_img_dir = out_path / "test_real" / "images"
        real_lbl_dir = out_path / "test_real" / "labels"
        copy_labeled_items(test_labeled, "test_real", real_img_dir, real_lbl_dir, tag="orig_abnormal")
        copy_normal_items(
            normal_by_split["test"],
            "test_real",
            real_img_dir,
            real_lbl_dir,
            len(normal_by_split["test"]),
            tag="all_original_normal",
        )
        print(
            "Created test_real: abnormal=",
            len(test_labeled),
            "normal=",
            len(normal_by_split["test"]),
            "augmentation=0",
        )

    # Normal-only test for false-positive counting.
    if CREATE_NORMAL_ONLY_TEST:
        normal_img_dir = out_path / "test_normal" / "images"
        normal_lbl_dir = out_path / "test_normal" / "labels"
        copy_normal_items(
            normal_by_split["test"],
            "test_normal",
            normal_img_dir,
            normal_lbl_dir,
            len(normal_by_split["test"]),
            tag="normal_only",
        )
        print("Created test_normal: normal=", len(normal_by_split["test"]))

    yaml_text = f"""path: {out_path}
train: train/images
val: val/images
test: test/images

nc: 2
names:
  0: dirt
  1: damage
"""
    data_yaml.write_text(yaml_text, encoding="utf-8")

    real_yaml_text = f"""path: {out_path}
train: train/images
val: val/images
test: test_real/images

nc: 2
names:
  0: dirt
  1: damage
"""
    real_yaml.write_text(real_yaml_text, encoding="utf-8")

    write_dataset_summary(out_path)

    print("")
    print("Recall-focused dataset created:", out_path)
    print("data.yaml:", data_yaml)
    print("data_real_test.yaml:", real_yaml)

    return str(data_yaml), str(real_yaml), len(list((out_path / "test" / "images").glob("*")))


# ============================================================
# 8. Plot training curves
# ============================================================

def resolve_train_save_dir(model):
    trainer = getattr(model, "trainer", None)
    save_dir = getattr(trainer, "save_dir", None)
    if save_dir is not None:
        return Path(save_dir)
    return Path(PROJECT_DIR) / RUN_NAME


def find_checkpoint(actual_save_dir, filename):
    candidates = [
        Path(actual_save_dir) / "weights" / filename,
        Path(PROJECT_DIR) / RUN_NAME / "weights" / filename,
        Path(PROJECT_DIR) / "detect" / RUN_NAME / "weights" / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


def find_resume_checkpoint():
    """Return an existing last.pt checkpoint for automatic resume, or None."""
    if RESUME_CHECKPOINT_PATH:
        p = Path(RESUME_CHECKPOINT_PATH).expanduser().resolve()
        return p if p.exists() else None

    candidates = [
        Path(PROJECT_DIR) / RUN_NAME / "weights" / "last.pt",
        Path(PROJECT_DIR) / "detect" / RUN_NAME / "weights" / "last.pt",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def get_completed_epoch_count(run_dir):
    """Best-effort count of completed epochs from results.csv."""
    try:
        import pandas as pd
        results_csv = Path(run_dir) / "results.csv"
        if not results_csv.exists():
            return 0
        df = pd.read_csv(results_csv)
        return len(df)
    except Exception:
        return 0


def normalize_results_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def find_existing_columns(df, candidates):
    return [c for c in candidates if c in df.columns]


def draw_curve(df, columns, title, ylabel, save_path):
    import matplotlib.pyplot as plt

    if not columns:
        print(f"Skipping graph because columns were not found: {title}")
        return None

    x = df["epoch"] if "epoch" in df.columns else np.arange(len(df))

    plt.figure(figsize=(10, 6))
    for col in columns:
        plt.plot(x, df[col], label=col)
    plt.title(title)
    plt.xlabel("epoch")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.show()
    print("Saved graph:", save_path)
    return save_path


def plot_training_curves(actual_save_dir):
    import pandas as pd

    actual_save_dir = Path(actual_save_dir)
    results_csv = actual_save_dir / "results.csv"

    if not results_csv.exists():
        candidates = [
            Path(PROJECT_DIR) / RUN_NAME / "results.csv",
            Path(PROJECT_DIR) / "detect" / RUN_NAME / "results.csv",
        ]
        for p in candidates:
            if p.exists():
                results_csv = p
                break

    if not results_csv.exists():
        print("results.csv not found. Cannot draw custom training graphs.")
        return []

    df = pd.read_csv(results_csv)
    df = normalize_results_columns(df)

    graph_dir = actual_save_dir / "custom_graphs"
    graph_dir.mkdir(parents=True, exist_ok=True)

    train_loss_cols = find_existing_columns(df, [
        "train/box_loss",
        "train/cls_loss",
        "train/dfl_loss",
    ])

    val_loss_cols = find_existing_columns(df, [
        "val/box_loss",
        "val/cls_loss",
        "val/dfl_loss",
    ])

    precision_recall_cols = find_existing_columns(df, [
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/precision",
        "metrics/recall",
    ])

    map_cols = find_existing_columns(df, [
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
        "metrics/mAP50",
        "metrics/mAP50-95",
    ])

    saved = []
    for saved_path in [
        draw_curve(df, train_loss_cols, "Train Loss Curves", "loss", graph_dir / "01_train_loss_curves.png"),
        draw_curve(df, val_loss_cols, "Validation Loss Curves", "loss", graph_dir / "02_val_loss_curves.png"),
        draw_curve(df, precision_recall_cols, "Precision and Recall Curves", "score", graph_dir / "03_precision_recall_curves.png"),
        draw_curve(df, map_cols, "mAP50 and mAP50-95 Curves", "mAP", graph_dir / "04_map_curves.png"),
    ]:
        if saved_path is not None:
            saved.append(saved_path)

    print("")
    print("Custom graph output folder:", graph_dir)
    return saved


# ============================================================
# 9. Train / evaluate
# ============================================================

def clear_cuda_memory():
    """Best-effort GPU memory cleanup between evaluation stages."""
    try:
        import gc
        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def is_cuda_oom_error(exc):
    text = str(exc).lower()
    return "cuda out of memory" in text or "outofmemoryerror" in text


def export_training_artifacts(actual_save_dir=None, note=""):
    """Copy the important outputs to a small export folder and zip it.

    This intentionally does NOT copy the generated dataset images, because that can be huge.
    It exports the model weights, graphs, training CSV/YAML files, and dataset summary.
    """
    if not EXPORT_ARTIFACTS:
        return None

    export_dir = Path(EXPORT_DIR)
    export_dir.mkdir(parents=True, exist_ok=True)

    candidate_run_dirs = []
    if actual_save_dir is not None:
        candidate_run_dirs.append(Path(actual_save_dir))
    candidate_run_dirs.extend([
        Path(PROJECT_DIR) / RUN_NAME,
        Path(PROJECT_DIR) / "detect" / RUN_NAME,
    ])

    run_dir = None
    for p in candidate_run_dirs:
        if p.exists():
            run_dir = p
            break

    copied = []

    def copy_file(src, dst_name=None):
        src = Path(src)
        if not src.exists() or not src.is_file():
            return
        dst = export_dir / (dst_name or src.name)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied.append(str(dst))

    def copy_tree(src, dst_name):
        src = Path(src)
        if not src.exists() or not src.is_dir():
            return
        dst = export_dir / dst_name
        shutil.copytree(src, dst, dirs_exist_ok=True)
        copied.append(str(dst))

    if run_dir is not None:
        copy_file(run_dir / "weights" / "best.pt", "best.pt")
        copy_file(run_dir / "weights" / "last.pt", "last.pt")
        copy_file(run_dir / "results.csv", "results.csv")
        copy_file(run_dir / "args.yaml", "args.yaml")
        copy_file(run_dir / "labels.jpg", "labels.jpg")
        copy_file(run_dir / "confusion_matrix.png", "confusion_matrix.png")
        copy_file(run_dir / "confusion_matrix_normalized.png", "confusion_matrix_normalized.png")
        copy_file(run_dir / "results.png", "results.png")
        copy_tree(run_dir / "custom_graphs", "custom_graphs")

    dataset_dir = Path(OUT)
    copy_file(dataset_dir / "dataset_summary.csv", "dataset_summary.csv")
    copy_file(dataset_dir / "data.yaml", "data.yaml")
    copy_file(dataset_dir / "data_real_test.yaml", "data_real_test.yaml")

    note_path = export_dir / "export_note.txt"
    note_path.write_text(
        f"note={note}\n"
        f"run_dir={run_dir}\n"
        f"dataset_dir={dataset_dir}\n"
        f"export_dir={export_dir}\n",
        encoding="utf-8",
    )

    zip_path = shutil.make_archive(EXPORT_ZIP_BASENAME, "zip", export_dir)
    print("")
    print("Artifact export")
    print("===============")
    print("Export folder:", export_dir)
    print("Export zip   :", zip_path)
    print("Copied items :", len(copied))
    return zip_path


def evaluate_normal_false_positives(model, dataset_out_path, device):
    if not RUN_NORMAL_FP_EVAL:
        return

    normal_dir = Path(dataset_out_path) / "test_normal" / "images"
    if not normal_dir.exists():
        print("Normal-only test folder not found. Skipping false-positive evaluation.")
        return

    images = []
    for ext in IMAGE_EXTS:
        images.extend(normal_dir.glob(f"*{ext}"))
        images.extend(normal_dir.glob(f"*{ext.upper()}"))
    images = sorted(set(images))

    if not images:
        print("No normal-only test images found. Skipping false-positive evaluation.")
        return

    rng = random.Random(SEED + 9090)
    rng.shuffle(images)
    if NORMAL_FP_EVAL_MAX_IMAGES is not None:
        images = images[:NORMAL_FP_EVAL_MAX_IMAGES]

    print("")
    print("Normal-only false-positive evaluation")
    print("=====================================")
    print("Images evaluated:", len(images))
    print("Confidence threshold:", NORMAL_FP_CONF)
    print("Batch size:", NORMAL_FP_BATCH_SIZE)

    images_with_detection = 0
    total_boxes = 0
    class_box_counts = Counter()
    evaluated_images = 0

    clear_cuda_memory()

    # Important: do not pass the full 1000-image list into model.predict at once.
    # Small chunks prevent CUDA OOM after training/validation stages.
    for start_idx in range(0, len(images), max(1, int(NORMAL_FP_BATCH_SIZE))):
        chunk = images[start_idx:start_idx + max(1, int(NORMAL_FP_BATCH_SIZE))]
        try:
            for result in model.predict(
                source=[str(p) for p in chunk],
                imgsz=IMGSZ,
                conf=NORMAL_FP_CONF,
                device=device,
                batch=max(1, int(NORMAL_FP_BATCH_SIZE)),
                stream=True,
                verbose=False,
            ):
                evaluated_images += 1
                boxes = getattr(result, "boxes", None)
                n = 0 if boxes is None else len(boxes)
                if n > 0:
                    images_with_detection += 1
                    total_boxes += n
                    try:
                        cls_list = boxes.cls.detach().cpu().numpy().astype(int).tolist()
                        class_box_counts.update(cls_list)
                    except Exception:
                        pass
        except RuntimeError as exc:
            if SKIP_NORMAL_FP_ON_OOM and is_cuda_oom_error(exc):
                print("CUDA OOM during optional normal-only false-positive evaluation.")
                print("This optional check is being stopped safely. Training weights/results are already saved.")
                clear_cuda_memory()
                break
            raise

        if evaluated_images % max(1, int(NORMAL_FP_CLEAR_CACHE_EVERY)) == 0:
            clear_cuda_memory()
            print(f"Normal FP progress: {evaluated_images}/{len(images)}")

    clear_cuda_memory()

    rate = images_with_detection / max(evaluated_images, 1)
    print("normal images evaluated:", evaluated_images)
    print("normal images with any detection:", images_with_detection)
    print("false-positive image rate:", round(rate, 4))
    print("total predicted boxes on normal images:", total_boxes)
    print("false-positive boxes by class:", dict(class_box_counts))

    # Save the normal-only check result as a small text file so it is included in the export zip.
    try:
        export_dir = Path(EXPORT_DIR)
        export_dir.mkdir(parents=True, exist_ok=True)
        fp_path = export_dir / "normal_false_positive_summary.txt"
        fp_path.write_text(
            "normal-only false-positive evaluation\n"
            f"images_requested={len(images)}\n"
            f"images_evaluated={evaluated_images}\n"
            f"confidence={NORMAL_FP_CONF}\n"
            f"batch_size={NORMAL_FP_BATCH_SIZE}\n"
            f"images_with_detection={images_with_detection}\n"
            f"false_positive_image_rate={rate:.6f}\n"
            f"total_predicted_boxes={total_boxes}\n"
            f"class_box_counts={dict(class_box_counts)}\n",
            encoding="utf-8",
        )
        print("Saved normal FP summary:", fp_path)
    except Exception as exc:
        print("Warning: failed to save normal FP summary:", exc)


def train_model(data_yaml, real_yaml, test_count):
    from ultralytics import YOLO

    device = get_device()

    print("")
    print("YOLOv8s recall-focused training")
    print("================================")
    print("Model architecture/checkpoint:", MODEL_ARCH)
    print("Run name:", RUN_NAME)
    print("Epochs:", EPOCHS)
    print("Image size:", IMGSZ)
    print("Batch:", BATCH)
    print("Freeze layers:", FREEZE_LAYERS)
    print("Early stop patience:", EARLY_STOP_PATIENCE)

    resume_path = find_resume_checkpoint()

    if AUTO_RESUME_IF_LAST_EXISTS and not FORCE_RESTART_TRAINING and resume_path is not None:
        completed_epochs = get_completed_epoch_count(resume_path.parent.parent)
        print("")
        print("Resume checkpoint found")
        print("=======================")
        print("last.pt:", resume_path)
        print("completed epochs found in results.csv:", completed_epochs)
        print("Action: resume training from last.pt, not from epoch 1")

        # Ultralytics resumes optimizer state, epoch, scheduler, and training args from last.pt.
        # Do not pass a full new train_kwargs dict here because resume=True should recover the saved run.
        model = YOLO(str(resume_path))
        model.train(resume=True)

    else:
        if FORCE_RESTART_TRAINING:
            print("")
            print("FORCE_RESTART_TRAINING=True, starting a new training run from:", MODEL_ARCH)
        else:
            print("")
            print("No last.pt checkpoint found. Starting a new training run from:", MODEL_ARCH)

        model = YOLO(MODEL_ARCH)

        train_kwargs = dict(
            data=data_yaml,
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            device=device,
            seed=SEED,
            project=PROJECT_DIR,
            name=RUN_NAME,
            exist_ok=True,
            workers=WORKERS,
            cache="disk",
            amp=(device != "cpu"),

            # More stable than optimizer=auto for this small-defect recall-focused setup.
            optimizer="AdamW",
            lr0=0.002,
            lrf=0.05,
            warmup_epochs=3.0,
            weight_decay=0.0005,

            patience=EARLY_STOP_PATIENCE,
            save=True,
            save_period=SAVE_PERIOD,
            plots=True,
            freeze=FREEZE_LAYERS,

            # The script already creates many augmented defect images.
            # Keep YOLO's internal augmentations gentler to avoid over-distorting small dirt/damage boxes.
            mosaic=0.3,
            close_mosaic=10,
            mixup=0.0,
            copy_paste=0.0,
            erasing=0.0,
            auto_augment=None,
            hsv_h=0.01,
            hsv_s=0.30,
            hsv_v=0.25,
            translate=0.05,
            scale=0.30,
            degrees=0.0,
            shear=0.0,
            perspective=0.0,

            cos_lr=True,
        )

        model.train(**train_kwargs)

    actual_save_dir = resolve_train_save_dir(model)
    best_path = find_checkpoint(actual_save_dir, "best.pt")
    last_path = find_checkpoint(actual_save_dir, "last.pt")

    print("")
    print("Training checkpoints")
    print("====================")
    print("actual save_dir:", actual_save_dir)
    print("best.pt:", best_path if best_path.exists() else "not found")
    print("last.pt:", last_path if last_path.exists() else "not found")

    print("")
    print("Drawing custom training graphs")
    print("==============================")
    plot_training_curves(actual_save_dir)

    # Export immediately after training/checkpoint creation.
    # This keeps best.pt/last.pt/results/graphs safe before any optional evaluation can fail.
    export_training_artifacts(actual_save_dir, note="after_training_before_final_eval")
    clear_cuda_memory()

    if test_count > 0 and best_path.exists():
        print("")
        print("Final balanced original-only test evaluation with trained best.pt")
        print("===============================================================")
        eval_model = YOLO(str(best_path))
        metrics = eval_model.val(
            data=data_yaml,
            split="test",
            device=device,
            imgsz=IMGSZ,
            batch=16 if BATCH == -1 else BATCH,
            workers=WORKERS,
            plots=True,
        )
        print("balanced test mAP50:", round(float(metrics.box.map50), 4))
        print("balanced test mAP50-95:", round(float(metrics.box.map), 4))
        export_training_artifacts(actual_save_dir, note="after_balanced_test_eval")
        clear_cuda_memory()

        if CREATE_REAL_DISTRIBUTION_TEST and Path(real_yaml).exists():
            print("")
            print("Final real-distribution original-only test evaluation")
            print("====================================================")
            real_metrics = eval_model.val(
                data=real_yaml,
                split="test",
                device=device,
                imgsz=IMGSZ,
                batch=16 if BATCH == -1 else BATCH,
                workers=WORKERS,
                plots=True,
            )
            print("real-distribution test mAP50:", round(float(real_metrics.box.map50), 4))
            print("real-distribution test mAP50-95:", round(float(real_metrics.box.map), 4))
            export_training_artifacts(actual_save_dir, note="after_real_distribution_test_eval")
            clear_cuda_memory()

        try:
            evaluate_normal_false_positives(eval_model, OUT, device)
        except RuntimeError as exc:
            if SKIP_NORMAL_FP_ON_OOM and is_cuda_oom_error(exc):
                print("Normal-only false-positive evaluation failed with CUDA OOM, but training artifacts are safe.")
                clear_cuda_memory()
            else:
                raise
        finally:
            export_training_artifacts(actual_save_dir, note="final_export")


# ============================================================
# 10. Main
# ============================================================

def main():
    check_packages()
    data_yaml, real_yaml, test_count = prepare_recall_focused_dataset()
    train_model(data_yaml, real_yaml, test_count)
    print("Finished YOLOv8s recall-focused wind turbine training.")


if __name__ == "__main__":
    main()


# 두번째 모델 학습 과정

다시 처음부터 yolov8s 모델을 가져와 학습을 시작함  

처음부터 다시 기본 데이터와 증강 데이터를 학습하려니 시간이 많이 걸리는 점 발견함  

코드 갈아 엎고  

기본 데이터를 학습한 모델에 증강만 한 데이터를 추가 학습 시킴(파인 튜닝)  

그래도 원하는 재현율(0.7 이상) 이 나오지 않아  

다시 코드 갈아 엎고  

train에 증강 데이터 90% 와 정상 데이터 10%의 분포로 학습 시킴  

GPU 없는 상태의 학습 속도는 지옥임  


![](./img/train_loss.png)

41~42 에포크를 지나면서 loss 값이 어느정도 수렴하는 결과를 확인함  

![](./img/validation_loss.png)

35 에포크 정도의 수준에서 점점 머리를 드는 차트 모습이다  

![](./img/recall.png)

precicion 과 recall 값이 30 에포크를 지나면서 어느정도 안정감을  유지하나 싶었지만  

점점 변동폭을 키우는걸 확인함  

![](./img/map50.png)

mAP50 과 mAP50 - 95 의 값은 에포크 30 수준에서부터 안정적으로 수렴하는걸 볼수 잇다  

코드 실행 결과는 GPU를 무료로 사용할수 있는 LMS, colab, kaggle 을 돌며 튕기지 않는 곳에서 계속  

유목민 처럼 옮겨 다니며 코드를 실행하다 보니 결과 파일을 저장하지 못했다 